# Markov Decision Processes (MDPs) — A Working Example

**Acting as an AI Engineer:** this notebook builds a small but complete
Reinforcement Learning example from scratch (only `numpy`).

An **MDP** is the mathematical framework behind model-based RL. It is defined by:

- **S** — set of states (where the agent can be)
- **A** — set of actions (what the agent can do)
- **P(s' | s, a)** — transition probabilities (the *model*)
- **R(s, a, s')** — reward received
- **γ (gamma)** — discount factor for future rewards (0 = greedy, 1 = far-sighted)

The goal is to find an **optimal policy** π*(s) that maximizes expected discounted return:

```
V*(s) = max_a  sum_{s'} P(s'|s,a) [ R(s,a,s') + γ V*(s') ]      (Bellman optimality)
```

We will use two classic *planning* algorithms that assume the model P is known:
**Value Iteration** and **Policy Iteration**.

## 1. The environment: a 4×4 Gridworld

A robot starts in the grid and must reach the **Goal** `G` (+1) while avoiding the
**Hole** `X` (-1). Every move costs nothing, but the world is *stochastic*: the
intended action succeeds with probability `0.8`, and slips left/right with `0.1`
each — so the agent must plan for uncertainty.

```
  (0,0)  (0,1)  (0,2)  (0,3)=G
  (1,0)  (1,1)=X  (1,2)  (1,3)
  (2,0)  (2,1)  (2,2)  (2,3)
  (3,0)  (3,1)  (3,2)  (3,3)
```

We build the transition model `P[s][a] = [(prob, next_state, reward, done), ...]`.


In [ ]:
import numpy as np

# ---- Gridworld definition ----
GRID_SIZE = 4
GOAL = (0, 3)      # reach the goal -> +1 reward, terminal
HOLE = (1, 1)      # fall in the hole -> -1 reward, terminal
GAMMA = 0.9        # discount factor
SLIP = 0.1         # probability of slipping to the side on each move

# Actions: 0=Up, 1=Right, 2=Down, 3=Left
ACTIONS = ["Up", "Right", "Down", "Left"]
DX = [-1, 0, 1, 0]
DY = [0, 1, 0, -1]
N_STATES = GRID_SIZE * GRID_SIZE
N_ACTIONS = 4


def coord_to_state(r, c):
    return r * GRID_SIZE + c


def state_to_coord(s):
    return divmod(s, GRID_SIZE)


def is_terminal(s):
    r, c = state_to_coord(s)
    return (r, c) == GOAL or (r, c) == HOLE


# Build the transition model P[s][a] = list of (prob, next_state, reward, done)
P = {s: {a: [] for a in range(N_ACTIONS)} for s in range(N_STATES)}
for s in range(N_STATES):
    r, c = state_to_coord(s)
    if is_terminal(s):
        # absorbing terminal state: stay put, no further reward
        for a in range(N_ACTIONS):
            P[s][a] = [(1.0, s, 0.0, True)]
        continue
    for a in range(N_ACTIONS):
        for delta_a, prob in [(0, 1.0 - 2 * SLIP), (-1, SLIP), (1, SLIP)]:
            aa = (a + delta_a) % N_ACTIONS          # intended / left-slip / right-slip
            nr, nc = r + DX[aa], c + DY[aa]
            if not (0 <= nr < GRID_SIZE and 0 <= nc < GRID_SIZE):
                nr, nc = r, c                        # bounce off the wall
            ns = coord_to_state(nr, nc)
            reward = 1.0 if (nr, nc) == GOAL else (-1.0 if (nr, nc) == HOLE else 0.0)
            done = (nr, nc) == GOAL or (nr, nc) == HOLE
            P[s][a].append((prob, ns, reward, done))

print(f"States: {N_STATES}, Actions: {N_ACTIONS}")
print("From state 0 (top-left), action Right ->", P[0][1])


## 2. Value Iteration

We iteratively apply the Bellman optimality update to the value function until it
stops changing. Then we read off the greedy policy."


In [ ]:
def value_iteration(P, gamma=GAMMA, theta=1e-6, max_iter=1000):
    V = {s: 0.0 for s in range(N_STATES)}
    for it in range(max_iter):
        delta = 0.0
        newV = {}
        for s in range(N_STATES):
            if is_terminal(s):
                newV[s] = 0.0
                continue
            best = -float("inf")
            for a in range(N_ACTIONS):
                q = sum(prob * (reward + gamma * V[ns])
                        for prob, ns, reward, _ in P[s][a])
                best = max(best, q)
            newV[s] = best
            delta = max(delta, abs(newV[s] - V[s]))
        V = newV
        if delta < theta:
            print(f"Value Iteration converged in {it + 1} iterations")
            break
    # greedy policy extraction
    policy = {}
    for s in range(N_STATES):
        if is_terminal(s):
            policy[s] = None
            continue
        best, ba = -float("inf"), None
        for a in range(N_ACTIONS):
            q = sum(prob * (reward + gamma * V[ns])
                    for prob, ns, reward, _ in P[s][a])
            if q > best:
                best, ba = q, a
        policy[s] = ba
    return V, policy


## 3. Policy Iteration

Alternate between (a) evaluating the current policy and (b) improving it greedily.
It usually converges in fewer sweeps than Value Iteration."


In [ ]:
def policy_evaluation(policy, P, gamma=GAMMA, theta=1e-6):
    V = {s: 0.0 for s in range(N_STATES)}
    while True:
        delta = 0.0
        for s in range(N_STATES):
            if is_terminal(s):
                continue
            a = policy[s]
            v = sum(prob * (reward + gamma * V[ns])
                    for prob, ns, reward, _ in P[s][a])
            delta = max(delta, abs(V[s] - v))
            V[s] = v
        if delta < theta:
            break
    return V


def policy_iteration(P, gamma=GAMMA):
    policy = {s: (0 if not is_terminal(s) else None) for s in range(N_STATES)}
    for _ in range(100):
        V = policy_evaluation(policy, P, gamma)
        stable = True
        for s in range(N_STATES):
            if is_terminal(s):
                continue
            best, ba = -float("inf"), None
            for a in range(N_ACTIONS):
                q = sum(prob * (reward + gamma * V[ns])
                        for prob, ns, reward, _ in P[s][a])
                if q > best:
                    best, ba = q, a
            if ba != policy[s]:
                policy[s] = ba
                stable = False
        if stable:
            break
    return V, policy


## 4. Run it and inspect the result

Both planners should agree on the optimal value function and policy."


In [ ]:
def render_value(V):
    grid = np.zeros((GRID_SIZE, GRID_SIZE))
    for s in range(N_STATES):
        r, c = state_to_coord(s)
        grid[r, c] = V[s]
    return np.round(grid, 3)


def render_policy(policy):
    sym = {0: "↑", 1: "→", 2: "↓", 3: "←"}
    rows = []
    for r in range(GRID_SIZE):
        row = []
        for c in range(GRID_SIZE):
            s = coord_to_state(r, c)
            if is_terminal(s):
                row.append("G" if (r, c) == GOAL else "X")
            else:
                row.append(sym[policy[s]])
        rows.append(" ".join(row))
    return "\n".join(rows)


V_vi, pi_vi = value_iteration(P)
V_pi, pi_pi = policy_iteration(P)

print("Optimal STATE VALUES (Value Iteration):")
print(render_value(V_vi))
print()
print("Optimal POLICY (arrows, G=goal, X=hole):")
print(render_policy(pi_vi))
print()
print("Value Iteration and Policy Iteration agree on values:",
      np.allclose(render_value(V_vi), render_value(V_pi)))


## 5. Visualize the value landscape (optional)

If `matplotlib` is installed we draw a heatmap of the optimal values."


In [ ]:
try:
    import matplotlib.pyplot as plt

    fig, ax = plt.subplots(figsize=(5, 4))
    im = ax.imshow(render_value(V_vi), cmap="viridis")
    ax.set_title("Optimal state values (γ=%.1f)" % GAMMA)
    for s in range(N_STATES):
        r, c = state_to_coord(s)
        ax.text(c, r, render_policy(pi_vi).splitlines()[r].split()[c],
                ha="center", va="center", color="white", fontsize=12)
    fig.colorbar(im, ax=ax)
    plt.show()
except ImportError:
    print("matplotlib not installed - skipping the heatmap (values above are still valid).")


## 6. Takeaways

- An **MDP** = (S, A, P, R, γ). Solving it = finding the optimal policy π*.
- **Value Iteration** bootstraps values with the Bellman optimality equation.
- **Policy Iteration** evaluates then improves a policy; often fewer sweeps.
- Both need the *model* P — this is **model-based** RL.
- When P is **unknown**, use **model-free** methods (e.g. Q-Learning / SARSA) which
  learn directly from sampled transitions. That is the natural next step after this
  notebook.

Try changing `GAMMA` (e.g. 0.7 vs 0.99) or `SLIP` (0.0 = deterministic) and rerun to
see how the optimal policy reacts."
